# Exploration of LAUS Unemployment Data
 
Data documentation notes available online at: https://download.bls.gov/pub/time.series/la/la.txt
Notes from the documentation:
- "Rates and ratios are expressed as percents with one
decimal place."
- location, survey series both in the series_id
    - la.area file contains the location codes
    - la.series file contains the series codes I *think*


Important codes:
- Unemployment rate measure code: 03
- area_type for counties and equivalents: F
- Period M13 is annual average, the rest are the months in order




In [ ]:
# # SETUP
# import pandas as pd

# laus = pd.read_csv(
#     "../data_raw/LAUS/la.data.64.County",
#     sep="\t"
# )

# area = pd.read_csv(
#     "../data_raw/LAUS/la.area",
#     sep="\t"
# )

# series = pd.read_csv(
#     "../data_raw/LAUS/la.series",
#     sep="\t"
# )

# measure = pd.read_csv(
#     "../data_raw/LAUS/la.measure",
#     sep="\t"
# )

# area_type = pd.read_csv(
#     "../data_raw/LAUS/la.area_type",
#     sep="\t"
# )

# period = pd.read_csv(
#     '../data_raw/LAUS/la.period',
#     sep="\t"
# )

## Main Data Overview + Pre-Clean

### Note:

During my exploratory analysis, I parsed the series_id field directly to better understand the structure of the LAUS identifiers and verify how survey, area, and measure information are encoded. After reviewing the full LAUS documentation in more detail, I realized the la.series and la.area files provide this metadata directly and are meant to be merged with the main data files for this exact purpose. I will be using the offical metadata files in my official cleaning file, but I kept this here to document my exploratory process!

In [3]:
# Checking dataset format
print(laus.head())
print('#################')
print(laus.columns)

# Cleaning up column names
print('#################')
laus.columns = laus.columns.str.strip()


   series_id                       year period         value footnote_codes
0  LAUCN010010000000003            1990    M01           6.5            NaN
1  LAUCN010010000000003            1990    M02           6.4            NaN
2  LAUCN010010000000003            1990    M03           5.6            NaN
3  LAUCN010010000000003            1990    M04           6.6            NaN
4  LAUCN010010000000003            1990    M05           6.1            NaN
#################
Index(['series_id                     ', 'year', 'period', '       value',
       'footnote_codes'],
      dtype='str')
#################


In [4]:
laus_2 = laus.copy()
# Separating series_id into its respective components according to documentation
laus_2['survey'] = laus_2['series_id'].str[:2]
laus_2['adjusted'] = laus_2['series_id'].str[2:3]
laus_2['area_code'] = laus_2['series_id'].str[3:18]
laus_2['measure'] = laus_2['series_id'].str[18:]

print('#################################')
print(laus_2.head())

#################################
                        series_id  year period         value footnote_codes  \
0  LAUCN010010000000003            1990    M01           6.5            NaN   
1  LAUCN010010000000003            1990    M02           6.4            NaN   
2  LAUCN010010000000003            1990    M03           5.6            NaN   
3  LAUCN010010000000003            1990    M04           6.6            NaN   
4  LAUCN010010000000003            1990    M05           6.1            NaN   

  survey adjusted        area_code       measure  
0     LA        U  CN0100100000000  03            
1     LA        U  CN0100100000000  03            
2     LA        U  CN0100100000000  03            
3     LA        U  CN0100100000000  03            
4     LA        U  CN0100100000000  03            


## Preparing and Merging Metadata Files

In [5]:
# AREA FILE PREPARATION

# File cleanup
print(area.columns) # all set

# Restricting to county and equivalent codes
area = area[area['area_type_code'] == 'F']

# Separating county and state to use when merging geographic index in
area[['county', 'state']] = area['area_text'].str.split(', ', expand=True)


print(area.head())

Index(['area_type_code', 'area_code', 'area_text', 'display_level',
       'selectable', 'sort_sequence'],
      dtype='str')
     area_type_code        area_code           area_text  display_level  \
1208              F  CN0100100000000  Autauga County, AL              0   
1209              F  CN0100300000000  Baldwin County, AL              0   
1210              F  CN0100500000000  Barbour County, AL              0   
1211              F  CN0100700000000     Bibb County, AL              0   
1212              F  CN0100900000000   Blount County, AL              0   

     selectable  sort_sequence          county state  
1208          T             33  Autauga County    AL  
1209          T             34  Baldwin County    AL  
1210          T             35  Barbour County    AL  
1211          T             36     Bibb County    AL  
1212          T             37   Blount County    AL  


In [6]:
# SERIES FILE PREPARATION

# Cleaning up file columns
series.columns = series.columns.str.strip()

# Restricting to county/county-equivalent subset
series = series[series['area_type_code'] == 'F']
print(series.head())

                           series_id area_type_code        area_code  \
1186  LAUCN010010000000003                        F  CN0100100000000   
1187  LAUCN010010000000004                        F  CN0100100000000   
1188  LAUCN010010000000005                        F  CN0100100000000   
1189  LAUCN010010000000006                        F  CN0100100000000   
1190  LAUCN010030000000003                        F  CN0100300000000   

      measure_code seasonal  srd_code  \
1186             3        U         1   
1187             4        U         1   
1188             5        U         1   
1189             6        U         1   
1190             3        U         1   

                                   series_title footnote_codes  begin_year  \
1186  Unemployment Rate: Autauga County, AL (U)            NaN        1990   
1187       Unemployment: Autauga County, AL (U)            NaN        1990   
1188         Employment: Autauga County, AL (U)            NaN        1990   
1189    

In [7]:
# Merging metadata files
laus = pd.merge(laus, series, on=('series_id'))

## Exploring and Cleaning Data + Metadata File
Importing file as cleaning in unemployment_cleaning_WIP.py


In [12]:
# Setup
complete = pd.read_csv('../data_intermediate/unemployment_preclean.csv')

# Inspection
print(complete.describe(include='all'))
print('#############################')
print(complete.info())


                             series_id          year   period         value  \
count                          1509693  1.509693e+06  1509693       1509693   
unique                            3225           NaN       13           432   
top     LAUCN060370000000003                     NaN      M01           4.0   
freq                               471           NaN   119103         28211   
mean                               NaN  2.007546e+03      NaN           NaN   
std                                NaN  1.041619e+01      NaN           NaN   
min                                NaN  1.990000e+03      NaN           NaN   
25%                                NaN  1.999000e+03      NaN           NaN   
50%                                NaN  2.008000e+03      NaN           NaN   
75%                                NaN  2.017000e+03      NaN           NaN   
max                                NaN  2.026000e+03      NaN           NaN   

       footnote_codes_x area_type_code_x        are

### Notes
- It appears sort_sequence OR srd_code might be state or county fips, so check that when merging in geo id file
- Consider bringing back area_text for graphing

In [20]:
# Checking States
print(complete['state'].unique())
    # There is one nan state, and while we have PR we do not have DC. Maybe nan is DC?

print('################################')

print(complete[complete['state'].isna()].head(20)) # looks like DC

print('################################')

print(complete[complete['state'].isna()]['county'].unique()) # it is DC! I will add DC as the state abbreviation to match the other files



<StringArray>
['AL', 'AK', 'AZ', 'AR', 'CA', 'CO', 'CT', 'DE',  nan, 'FL', 'GA', 'HI', 'ID',
 'IL', 'IN', 'IA', 'KS', 'KY', 'LA', 'ME', 'MD', 'MA', 'MI', 'MN', 'MS', 'MO',
 'MT', 'NE', 'NV', 'NH', 'NJ', 'NM', 'NY', 'NC', 'ND', 'OH', 'OK', 'OR', 'PA',
 'RI', 'SC', 'SD', 'TN', 'TX', 'UT', 'VT', 'VA', 'WA', 'WV', 'WI', 'WY', 'PR']
Length: 52, dtype: str
################################
                             series_id  year period         value  \
149589  LAUCN110010000000003            1990    M01           5.5   
149590  LAUCN110010000000003            1990    M02           5.7   
149591  LAUCN110010000000003            1990    M03           5.4   
149592  LAUCN110010000000003            1990    M04           5.3   
149593  LAUCN110010000000003            1990    M05           5.6   
149594  LAUCN110010000000003            1990    M06           6.2   
149595  LAUCN110010000000003            1990    M07           6.3   
149596  LAUCN110010000000003            1990    M08           

In [26]:
# Pre-aggregation Checks
df = pd.read_csv('../data_clean/unemployment_WIP.csv')

categoricals = ['year', 'period', 'footnote_codes_x', 'county', 'state']

big_categoricals = ['area_code', 'county', 'srd_code', 'sort_sequence']

for category in categoricals:
    print(category + ' unique values:')
    print(df[category].unique())
    print('-------------------')

print('#######################')
for category in big_categoricals:
    print(category + ' number of unique values:')
    print(df[category].nunique())
    print('-------------------')


year unique values:
[1990 1991 1992 1993 1994 1995 1996 1997 1998 1999 2000 2001 2002 2003
 2004 2005 2006 2007 2008 2009 2010 2011 2012 2013 2014 2015 2016 2017
 2018 2019 2020 2021 2022 2023 2024 2025 2026]
-------------------
period unique values:
<StringArray>
['M01', 'M02', 'M03', 'M04', 'M05', 'M06', 'M07', 'M08', 'M09', 'M10', 'M11',
 'M12']
Length: 12, dtype: str
-------------------
footnote_codes_x unique values:
<StringArray>
[nan, 'X', 'P', 'R', 'N', 'Y']
Length: 6, dtype: str
-------------------
county unique values:
<StringArray>
[         'Autauga County',          'Baldwin County',
          'Barbour County',             'Bibb County',
           'Blount County',          'Bullock County',
           'Butler County',          'Calhoun County',
         'Chambers County',         'Cherokee County',
 ...
      'Toa Alta Municipio',      'Toa Baja Municipio',
 'Trujillo Alto Municipio',        'Utuado Municipio',
     'Vega Alta Municipio',     'Vega Baja Municipio',
      

#### Notes: MYSTERIOUS NUMBER OF COUNTIES

- It looks like there might be an issue with the county variable as there are only 1963 unique county names but 3225 distinct area codes
    - Check unmatched carefully upon merging with fips file
- It looks like srd_code might be state fips (52 unique values with DC and PR)
- It looks like sort_sequence might be county_fips (3225 unique values)
- By a google search, there should be 3,244 counties in the US including DC and territory municipalities 
    - this would have to include american samoa, which I haven't seen in the list of states :/
    - this means there is one extra county beyond the mystery american samoa counties
    